# Pavana — Pemodelan PM2.5 Kubu Raya

Eksperimen retrospektif prediksi **PM2.5 CAMS pada t+6 jam** di satu sel grid sekitar Sungai Raya,
Kubu Raya. Model tabular LightGBM dibandingkan dengan persistence dan rata-rata musiman,
kemudian dianalisis menggunakan SHAP.

**Cakupan:** target berupa keluaran CAMS, bukan pengukuran stasiun. Cuaca berasal dari reanalisis
ERA5. Evaluasi ini belum mengukur akurasi terhadap observasi atau skill prakiraan operasional.

## Konfigurasi

Data dan kode training menggunakan commit terpisah yang dipin dan diverifikasi terhadap manifest.
Lingkungan eksekusi serta artifact dicatat bersama hasil. Runner menggunakan CPU; dependensi
diinstal pada runtime. Parameter `SMOKE` membedakan pemeriksaan eksekusi dari eksperimen penuh.

In [ ]:
"""Cloud runner: pins, writable directories, and the switch for a smoke run."""

import json
import os
import sys
from pathlib import Path

REPO_ID = "myudak/pavana"

# Two separate pins. Never `main`: a later upload must not change what a recorded result was
# computed from. Data PR #5 was merged on 2026-10-15 and `main` carries the same bytes (verified
# against this manifest), but the experiment keeps the commit its CP1-CP3 evidence was hashed on.
DATA_REF = "main"
DATA_REVISION = "aaa0d2714102b7d3a7b3ad0beb2b1da1ce6a6dbe"
CODE_REF = "refs/pr/6"
CODE_REVISION = "a9a0d6d98008e08e80d6e978d82cfc89f79282af"  # commit that carries this bundle
RESOLVE_CODE_HEAD = False  # True resolves CODE_REF once and installs that commit instead

RUN_VERSION = "cams-t6h-retro-kaggle-explain-v1"
SMOKE = False  # True: 30 rounds. An execution check, never a reported result.
MAX_ROUNDS = 30 if SMOKE else 500
RUN_VERSION_EFFECTIVE = f"{RUN_VERSION}-smoke" if SMOKE else RUN_VERSION


def cloud_work_root() -> Path:
    """A writable directory: /kaggle/working, /content, or an override for local checks."""
    override = os.environ.get("PAVANA_CLOUD_WORK_ROOT")
    if override:
        return Path(override).expanduser()
    for candidate in ("/kaggle/working", "/content"):
        if Path(candidate).is_dir() and os.access(candidate, os.W_OK):
            return Path(candidate) / "pavana"
    return Path.cwd() / "cloud-run"


WORK_ROOT = cloud_work_root()
DATA_DIR = WORK_ROOT / "data"
ARTIFACTS_DIR = WORK_ROOT / "artifacts"
CODE_ROOT = WORK_ROOT / "code-download"
for path in (DATA_DIR, ARTIFACTS_DIR, CODE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

# `pavana_ml.paths` reads these when it is imported, so they must be set before the first import.
os.environ["PAVANA_ML_DATA_DIR"] = str(DATA_DIR)
os.environ["PAVANA_ML_ARTIFACTS_DIR"] = str(ARTIFACTS_DIR)

already = sorted(name for name in sys.modules if name.split(".")[0] == "pavana_ml")
print("python         :", sys.version.split()[0])
print("work root      :", WORK_ROOT, "| writable:", os.access(WORK_ROOT, os.W_OK))
print("data dir       :", DATA_DIR)
print("artifacts dir  :", ARTIFACTS_DIR)
print("pavana_ml already imported in this kernel:", already or "no")
if already:
    raise RuntimeError("Restart kernel/sesi sebelum Run all: pavana_ml sudah diimpor.")

## Data dan reproduksibilitas

Snapshot memuat deret CAMS dan ERA5 pada grid masing-masing, deteksi FIRMS, serta snapshot BMKG
sebagai konteks. Pipeline memakai deret per jam dan fitur deteksi di sekitar anchor Kubu Raya.
BMKG tidak digabungkan menjadi target historis. Timestamp kanonik menggunakan UTC.

Manifest mengidentifikasi byte snapshot; provenance sumber dan versi library tersimpan dalam
laporan audit serta `cloud_run.json`. Tidak ada API key penyedia data yang diperlukan.

In [ ]:
import importlib.util
import subprocess


def pip(*args: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)


if importlib.util.find_spec("huggingface_hub") is None:
    print("huggingface_hub is missing; installing it from PyPI")
    pip("install", "--quiet", "huggingface-hub>=0.26")

from huggingface_hub import HfApi  # noqa: E402

api = HfApi()
for label, ref in (("code", CODE_REF), ("data", DATA_REF)):
    info = api.dataset_info(REPO_ID, revision=ref)
    print(f"{label} ref  :", ref, "->", info.sha)
print("Internet works and the dataset is readable. No GPU and no provider API key is needed.")

In [ ]:
import hashlib

from huggingface_hub import snapshot_download


def file_sha256(path: Path) -> str:
    """Hash the downloaded bytes with the same function CODE_MANIFEST.json was built with."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


if RESOLVE_CODE_HEAD:
    CODE_REVISION = HfApi().dataset_info(REPO_ID, revision=CODE_REF).sha
    print("resolved", CODE_REF, "once for this run ->", CODE_REVISION)
else:
    print("pinned code revision ->", CODE_REVISION)

download = Path(
    snapshot_download(
        REPO_ID,
        repo_type="dataset",
        revision=CODE_REVISION,
        allow_patterns=["code/**"],
        local_dir=CODE_ROOT,
    )
)
bundle = download / "code"
manifest = json.loads((bundle / "CODE_MANIFEST.json").read_text(encoding="utf-8"))

mismatched = []
for entry in manifest["files"]:
    path = bundle / entry["path"].removeprefix("code/")
    if not path.is_file() or file_sha256(path) != entry["sha256"]:
        mismatched.append(entry["path"])
assert not mismatched, f"bundle does not match CODE_MANIFEST.json: {mismatched}"

print("bundle root :", bundle)
print(f"verified {len(manifest['files'])} files against CODE_MANIFEST.json at {CODE_REVISION[:12]}")
print("The data revision is a separate pin:", DATA_REVISION)

In [ ]:
pip("install", "--quiet", "--upgrade", f"{bundle / 'ml'}[hub]")

import pavana_ml  # noqa: E402
from pavana_ml.paths import ARTIFACTS_DIR as RESOLVED_ARTIFACTS  # noqa: E402
from pavana_ml.paths import DATA_DIR as RESOLVED_DATA  # noqa: E402

print("package       :", Path(pavana_ml.__file__).parent)
print("DATA_DIR      :", RESOLVED_DATA)
print("ARTIFACTS_DIR :", RESOLVED_ARTIFACTS)
assert Path(RESOLVED_DATA) == DATA_DIR, RESOLVED_DATA
assert Path(RESOLVED_ARTIFACTS) == ARTIFACTS_DIR, RESOLVED_ARTIFACTS
print("The installed package resolves the writable cloud directories, not repository defaults.")

## Audit dan feature engineering

Deret diselaraskan per jam. Konsentrasi memakai µg/m³, kecepatan angin m/s, dan arah angin
meteorologis FROM. Nilai hilang dipertahankan sebagai NaN. Deteksi satelit bukan konfirmasi
kebakaran; ketiadaan deteksi tidak membuktikan ketiadaan kebakaran.

Fitur meliputi riwayat CAMS, cuaca, kalender, jumlah deteksi/FRP pada beberapa radius dan jendela
waktu, serta upwind pada varian E. Target t+6 tidak masuk ke fitur.

In [ ]:
from pavana_ml.data.research import MANIFEST_PATHS, download_snapshot

curated = sorted(MANIFEST_PATHS.values())
receipt_path = DATA_DIR / "dataset_snapshot.json"
missing = [path for path in curated if not (DATA_DIR / path).exists()]
if missing or not receipt_path.exists():
    print(f"downloading the {len(curated)} curated files at {DATA_REVISION[:12]}")
    print("the raw FIRMS batches stay on the Hub: the model table only reads the curated parquet")
    download_snapshot(DATA_DIR, DATA_REVISION, include=curated)
else:
    print("curated snapshot already present in this runtime; nothing re-downloaded")

stored = json.loads(receipt_path.read_text(encoding="utf-8"))
print("requested     :", stored["requested_revision"])
print("resolved      :", stored["resolved_revision"])
print("manifest sha  :", stored["manifest_sha256"])
print("verified files:", len(stored["verified_files"]))
assert stored["resolved_revision"] == DATA_REVISION, stored["resolved_revision"]
print("Every downloaded file matched the SHA256 recorded in the manifest at that commit.")

In [ ]:
from pavana_ml.data.research import prepare_cached_data, save_prepared

hourly, hotspots, report = prepare_cached_data(DATA_DIR)
audit = save_prepared(hourly, report, DATA_DIR)
assert report["aligned"]["rows"] == 24_432
assert report["snapshot"]["revision_verified"]

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

data_audit = report
source_roles = {
    "cams": "Keluaran model CAMS; target eksperimen, bukan observasi",
    "weather": "Reanalisis ERA5; fitur, bukan prakiraan issue-time",
    "hotspots": "Deteksi satelit FIRMS; fitur, bukan kebakaran terkonfirmasi",
    "bmkg_pm25": "Observasi BMKG; konteks saja, arsip terlalu pendek",
    "bmkg_weather": "Snapshot prakiraan BMKG; konteks saja",
}
inventory = pd.DataFrame(
    [
        {"Sumber": name, "Baris": entry["rows"], "Peran": source_roles[name]}
        for name, entry in data_audit["sources"].items()
    ]
)
display(inventory)
display(
    Markdown(
        f"**Periode deret per jam (UTC):** {data_audit['aligned']['start_utc']} "
        f"sampai {data_audit['aligned']['end_utc']}. "
        "Koordinat sel grid asli dicatat dalam laporan audit."
    )
)
missing = pd.Series(data_audit["missing_by_column"], name="Jumlah NaN")
display(missing[missing > 0].rename_axis("Kolom").to_frame())
print("Nilai hilang tetap NaN; tidak diisi nol atau diinterpolasi diam-diam.")

In [ ]:
from pavana_ml.features.model_table import build_model_table, load_feature_inputs, save_model_table

hourly, hotspots, inputs = load_feature_inputs(DATA_DIR)
table, metadata = build_model_table(hourly, hotspots, inputs=inputs)
table_saved = save_model_table(table, metadata, DATA_DIR)
display(pd.DataFrame([metadata["eligibility"]]).T.rename(columns={0: "Jumlah baris"}))

## Metode dan evaluasi

| Varian | Input / metode |
| --- | --- |
| A | Persistence: CAMS pada t |
| B | Rata-rata target per bulan/jam, dihitung dari train |
| C | LightGBM: riwayat CAMS, cuaca, angin, kalender |
| D | C + jumlah deteksi hotspot dan FRP |
| E | D + indikator upwind |

Train mencakup 2024, validation 2025, dan test 2026, dengan jeda enam jam pada batas split.
Varian LightGBM dipilih melalui MAE validation. Semua varian dinilai pada baris test identik.
Baseline B bukan prakiraan CAMS issue-time; snapshot tidak menyediakan arsip waktu terbit.

In [ ]:
from pavana_ml.models.experiment import load_model_table, run_experiment

table, metadata = load_model_table(DATA_DIR)
run = run_experiment(
    table, metadata, max_rounds=MAX_ROUNDS, version=RUN_VERSION_EFFECTIVE, if_exists="reuse"
)
results, report = run["results"], run["report"]

## Hasil

Metrik berikut dihitung ulang dari prediksi tersimpan. Grafik error memakai seluruh holdout;
cuplikan deret waktu memakai 168 jam berurutan paling awal, tanpa pemilihan berdasarkan polusi
atau error. MAE/RMSE dinyatakan dalam µg/m³, bukan persentase akurasi.

In [ ]:
import numpy as np


def checked_demo_inputs(run_dir):
    """Baca hasil dari disk dan cocokkan metrik sebelum membuat klaim/grafik."""
    saved_report = json.loads((run_dir / "report.json").read_text(encoding="utf-8"))
    metrics = pd.read_csv(run_dir / "metrics.csv", float_precision="round_trip")
    metrics = metrics[(metrics["split"] == "test") & (metrics["period"] == "all")]
    if len(metrics) != 5 or set(metrics["variant"]) != set("ABCDE"):
        raise ValueError("Harus ada tepat satu baris test/all untuk masing-masing A–E.")
    metrics = metrics.set_index("variant").loc[list("ABCDE")]
    predictions = pd.read_parquet(run_dir / "test_predictions.parquet")
    for column in ("valid_at", "target_valid_at"):
        values = predictions[column]
        if values.dt.tz is None or str(values.dt.tz) != "UTC" or values.isna().any():
            raise ValueError("Timestamp prediksi harus UTC, lengkap, dan tidak ambigu.")
    predictions = predictions.sort_values("target_valid_at").reset_index(drop=True)
    if predictions["target_valid_at"].duplicated().any():
        raise ValueError("Timestamp target test tidak boleh duplikat.")
    if (
        not (predictions["target_valid_at"] - predictions["valid_at"])
        .eq(pd.Timedelta(hours=6))
        .all()
    ):
        raise ValueError("Horizon prediksi harus tepat enam jam.")
    status = saved_report["status"]
    if saved_report["target_kind"] != "model" or not status["research_only"]:
        raise ValueError("Demo ini khusus eksperimen retrospektif bertarget CAMS.")
    if status["operational_ready"] or status["observation_validated"]:
        raise ValueError("Demo CAMS tidak boleh mengklaim validasi observasi/operasional.")
    selected = saved_report["selected_variant"]
    validation = saved_report["validation_mae"]
    expected = min("CDE", key=lambda key: (validation[key], "CDE".index(key)))
    if selected != expected or saved_report["selected_on"] != "validation MAE":
        raise ValueError("Pilihan LightGBM harus berasal dari validation MAE saja.")
    for key in "ABCDE":
        actual = predictions["target"].to_numpy(dtype=float)
        predicted = predictions[f"pred_{key}"].to_numpy(dtype=float)
        if not np.isfinite(actual).all() or not np.isfinite(predicted).all():
            raise ValueError("Target/prediksi hilang: jangan menyusutkan baris evaluasi.")
        if int(metrics.loc[key, "n"]) != len(predictions):
            raise ValueError("Jumlah baris test harus sama untuk semua varian.")
        recalculated = {
            "mae": np.abs(actual - predicted).mean(),
            "rmse": np.sqrt(np.square(actual - predicted).mean()),
        }
        for metric, value in recalculated.items():
            recorded = saved_report["metrics"]["test"][key][metric]
            if not np.isclose(value, metrics.loc[key, metric], rtol=1e-12, atol=1e-12):
                raise ValueError(f"{key}: {metric} tidak cocok dengan file prediksi.")
            if not np.isclose(value, recorded, rtol=1e-12, atol=1e-12):
                raise ValueError(f"{key}: {metric} tidak cocok dengan report.json.")
    return saved_report, metrics, predictions


def first_holdout_week(predictions):
    """Pilih 168 jam berurutan paling awal; tidak memilih berdasarkan besar polusi/error."""
    groups = predictions["target_valid_at"].diff().ne(pd.Timedelta(hours=1)).cumsum()
    for _, block in predictions.groupby(groups, sort=False):
        if len(block) >= 168:
            return block.iloc[:168].copy()
    raise ValueError("Holdout belum memiliki 168 jam berurutan untuk grafik mingguan.")


def comparison_sentence(model_mae, baseline_mae):
    difference = model_mae - baseline_mae
    if np.isclose(difference, 0, rtol=0, atol=1e-9):
        return "MAE LightGBM terpilih setara dengan persistence pada holdout ini."
    direction = "lebih tinggi (lebih buruk)" if difference > 0 else "lebih rendah (lebih baik)"
    return f"MAE LightGBM terpilih {abs(difference):.2f} µg/m³ {direction} dari persistence."


def demo_takeaway(metrics, selected):
    winner = metrics["mae"].idxmin()
    comparison = comparison_sentence(metrics.loc[selected, "mae"], metrics.loc["A", "mae"])
    return (
        f"MAE test terendah di antara A–E: **{winner}**. "
        f"LightGBM **{selected}** dipilih berdasarkan validation, bukan test. {comparison} "
        "Hasil ini mengukur prediksi medan model CAMS, belum akurasi terhadap stasiun."
    )

In [ ]:
demo_report, demo_metrics, demo_predictions = checked_demo_inputs(run["output_dir"])
selected = demo_report["selected_variant"]
week = first_holdout_week(demo_predictions)
presentation_dir = run["output_dir"] / "presentation"
presentation_dir.mkdir(exist_ok=True)
demo_metrics.to_csv(presentation_dir / "holdout_metrics.csv", index_label="variant")
week.to_csv(presentation_dir / "first_holdout_week.csv", index=False)
display(
    demo_metrics[["n", "features_n", "mae", "rmse"]]
    .rename(
        columns={
            "n": "Jam test",
            "features_n": "Jumlah fitur",
            "mae": "MAE (µg/m³)",
            "rmse": "RMSE (µg/m³)",
        }
    )
    .round(2)
)
takeaway = demo_takeaway(demo_metrics, selected)
display(Markdown(takeaway))
if SMOKE:
    display(Markdown("**SMOKE RUN 30 ronde: hanya cek eksekusi, bukan hasil eksperimen penuh.**"))

In [ ]:
import matplotlib.pyplot as plt

labels = [
    "A · Persistence",
    "B · Rata-rata musiman",
    "C · LightGBM: riwayat + cuaca",
    "D · C + hotspot/FRP",
    "E · D + upwind",
]
style = {
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "figure.facecolor": "white",
}
scope = (
    f"{len(demo_predictions):,} jam test yang sama · target CAMS t+6 jam · "
    f"{demo_predictions['target_valid_at'].min():%d %b %Y}–"
    f"{demo_predictions['target_valid_at'].max():%d %b %Y} UTC"
)
with plt.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
    for ax, metric, title in zip(axes, ("mae", "rmse"), ("MAE", "RMSE"), strict=True):
        bars = ax.barh(labels, demo_metrics[metric], color="#3B82F6", height=0.6)
        bars[0].set_facecolor("#475569")
        ax.bar_label(bars, fmt="%.2f", padding=5)
        ax.set_xlim(0, float(demo_metrics[metric].max()) * 1.22)
        ax.invert_yaxis()
        ax.set_xlabel("Error (µg/m³) · lebih kecil lebih baik")
        ax.set_title(title, loc="left")
        ax.grid(axis="x", alpha=0.18)
        ax.set_axisbelow(True)
    axes[1].tick_params(axis="y", labelleft=False)
    fig.suptitle("Perbandingan error pada seluruh holdout", fontsize=16)
    fig.supxlabel(
        scope + "\nEksperimen retrospektif; belum divalidasi terhadap observasi stasiun.",
        fontsize=10,
    )
    fig.savefig(presentation_dir / "holdout_errors.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
display(Markdown("**Cara baca:** panjang batang menunjukkan error terhadap CAMS. " + takeaway))

In [ ]:
from zoneinfo import ZoneInfo

import matplotlib.dates as mdates

local_times = week["target_valid_at"].dt.tz_convert("Asia/Jakarta")
with plt.rc_context(style):
    fig, ax = plt.subplots(figsize=(12, 4.8), layout="constrained")
    for column, label, color, dash in (
        ("target", "Target CAMS", "#1F2937", "-"),
        ("pred_A", "A · Persistence", "#B45309", "--"),
        (f"pred_{selected}", f"{selected} · LightGBM (dipilih dari validation)", "#2563EB", "-."),
    ):
        ax.plot(local_times, week[column], label=label, color=color, linestyle=dash, linewidth=1.8)
    ticks = [local_times.iloc[0], *local_times.iloc[24:-1:24], local_times.iloc[-1]]
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M", tz=ZoneInfo("Asia/Jakarta")))
    ax.set_xlim(local_times.iloc[0], local_times.iloc[-1])
    ax.set_ylim(bottom=0)
    ax.set_ylabel("PM2.5 keluaran model (µg/m³)")
    ax.set_xlabel("Waktu target (WIB) · 168 jam berurutan paling awal di holdout")
    ax.set_title("Contoh minggu pertama: target CAMS dan prediksi t+6 jam", loc="left", pad=45)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), ncol=3, frameon=False, fontsize=10)
    ax.grid(alpha=0.18)
    fig.supxlabel(
        "Cuplikan dipilih berdasarkan waktu. Metrik dihitung pada seluruh holdout.",
        fontsize=10,
    )
    fig.savefig(presentation_dir / "first_holdout_week.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
display(
    Markdown(
        "Grafik memperlihatkan bentuk prediksi pada satu minggu; kesimpulan perbandingan memakai "
        "seluruh holdout pada tabel di atas. **Target CAMS bukan pengukuran stasiun.**"
    )
)

## Explainability — SHAP

Tree SHAP menjelaskan varian LightGBM yang dipilih melalui validation. Penjelasan global memakai
maksimal 1.000 baris holdout yang disampel dengan seed tetap, tanpa melihat target/error.
Penjelasan lokal memakai 168 jam berurutan paling awal. NaN tetap dipertahankan.

Metode `tree_path_dependent` menggunakan distribusi jalur pohon yang tersimpan saat training.
Nilai dasar dan jumlah kontribusi fitur harus merekonstruksi prediksi model dalam µg/m³.

In [ ]:
def build_research_explanations(run_dir, model_table, table_metadata, data_revision, code_revision):
    """Explain the evaluated artifact on aligned holdout rows; keep serving gates unchanged."""
    import hashlib

    import lightgbm as lgb
    import shap

    from pavana_ml.models.experiment import frame_fingerprint

    saved, metrics, predictions = checked_demo_inputs(run_dir)
    variant = saved["selected_variant"]
    artifact = run_dir / variant
    meta = json.loads((artifact / "metadata.json").read_text(encoding="utf-8"))
    model_hash = hashlib.sha256((artifact / "model.txt").read_bytes()).hexdigest()
    if (
        model_hash != meta["model_sha256"]
        or model_hash != saved["artifacts"][variant]["model_sha256"]
        or not meta["evaluated"]
        or meta["target_kind"] != "model"
        or meta["operational_ready"]
        or meta["observation_validated"]
    ):
        raise ValueError("Checksum/status model tidak cocok dengan artifact yang dievaluasi.")
    if frame_fingerprint(model_table) != saved["data"]["frame_sha256"]:
        raise ValueError("Tabel fitur tidak cocok dengan frame training yang tercatat.")
    features = meta["features"]
    if features != saved["feature_allowlists"][variant]:
        raise ValueError("Urutan fitur berbeda dari model yang dievaluasi.")
    rows = model_table.set_index("valid_at").loc[predictions["valid_at"]]
    x = rows[features]
    if not np.array_equal(
        rows[table_metadata["target"]].to_numpy(), predictions["target"].to_numpy()
    ):
        raise ValueError("Target tabel tidak sejajar dengan prediksi holdout.")
    model = lgb.Booster(model_file=str(artifact / "model.txt"))
    if not np.allclose(model.predict(x), predictions[f"pred_{variant}"], rtol=1e-10, atol=1e-10):
        raise ValueError("Model tersimpan tidak mereproduksi prediksi holdout.")
    week_rows = first_holdout_week(predictions)
    local_indices = pd.Index(predictions["valid_at"]).get_indexer(week_rows["valid_at"])
    seed = 20261015
    global_indices = np.sort(
        np.random.default_rng(seed).choice(len(x), min(1000, len(x)), replace=False)
    )
    indices = np.union1d(global_indices, local_indices)
    explained_x = x.iloc[indices]
    explanation = shap.TreeExplainer(
        model, feature_perturbation="tree_path_dependent", model_output="raw"
    )(explained_x, check_additivity=True)
    predicted = predictions[f"pred_{variant}"].iloc[indices].to_numpy()
    if not np.allclose(
        explanation.base_values + explanation.values.sum(axis=1), predicted, rtol=1e-7, atol=1e-6
    ):
        raise ValueError("Jumlah kontribusi SHAP tidak merekonstruksi prediksi.")
    positions = {int(index): position for position, index in enumerate(indices)}
    global_positions = [positions[int(index)] for index in global_indices]
    cases = []
    for index in local_indices:
        row = predictions.iloc[index]
        position = positions[int(index)]
        cases.append(
            {
                "originAt": row["valid_at"].isoformat(),
                "targetAt": row["target_valid_at"].isoformat(),
                "targetCamsUgM3": float(row["target"]),
                "predictionUgM3": float(row[f"pred_{variant}"]),
                "persistenceUgM3": float(row["pred_A"]),
                "baseValueUgM3": float(explanation.base_values[position]),
                "featureValues": [float(v) if np.isfinite(v) else None for v in x.iloc[index]],
                "shapValuesUgM3": explanation.values[position].tolist(),
            }
        )
    bundle = {
        "schemaVersion": 1,
        "runVersion": saved["version"],
        "modelVariant": variant,
        "modelSha256": model_hash,
        "datasetRevision": data_revision,
        "codeRevision": code_revision,
        "generatedAt": pd.Timestamp.now(tz="UTC").isoformat(),
        "location": {
            "name": "Sungai Raya, Kubu Raya",
            **table_metadata["target_location"],
            "timezone": "Asia/Jakarta",
        },
        "horizonHours": 6,
        "targetKind": "model",
        "researchOnly": True,
        "operationalReady": False,
        "observationValidated": False,
        "method": "tree_path_dependent",
        "shapVersion": shap.__version__,
        "features": features,
        "globalImportance": {
            "sampleN": len(global_indices),
            "seed": seed,
            "meanAbsShapUgM3": np.abs(explanation.values[global_positions]).mean(axis=0).tolist(),
        },
        "metrics": [
            {
                "variant": key,
                "maeUgM3": float(row["mae"]),
                "rmseUgM3": float(row["rmse"]),
                "n": int(row["n"]),
            }
            for key, row in metrics.iterrows()
        ],
        "cases": cases,
    }
    return bundle, explanation[global_positions], explanation[positions[int(local_indices[0])]]

In [ ]:
research_bundle, global_explanation, local_explanation = build_research_explanations(
    run["output_dir"], table, metadata, DATA_REVISION, CODE_REVISION
)
research_path = run["output_dir"] / "research.json"
research_path.write_text(
    json.dumps(research_bundle, allow_nan=False, separators=(",", ":")), encoding="utf-8"
)
display(
    pd.DataFrame(
        {
            "Fitur": research_bundle["features"],
            "Rerata |SHAP| (µg/m³)": research_bundle["globalImportance"]["meanAbsShapUgM3"],
        }
    )
    .sort_values("Rerata |SHAP| (µg/m³)", ascending=False)
    .head(10)
)

In [ ]:
import shap

for kind, explanation in (
    ("importance", global_explanation),
    ("beeswarm", global_explanation),
    ("waterfall", local_explanation),
):
    with plt.rc_context(style):
        if kind == "importance":
            shap.plots.bar(explanation, max_display=12, show=False)
            plt.xlabel("Rerata |SHAP| terhadap prediksi CAMS (µg/m³)")
        elif kind == "beeswarm":
            shap.plots.beeswarm(explanation, max_display=12, show=False)
            plt.xlabel("Kontribusi terhadap prediksi CAMS (µg/m³)")
        else:
            shap.plots.waterfall(explanation, max_display=12, show=False)
            plt.xlabel("Nilai dasar + kontribusi = prediksi CAMS (µg/m³)")
        fig = plt.gcf()
        fig.set_size_inches(12, 7)
        fig.savefig(presentation_dir / f"shap_{kind}.png", dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)

In [ ]:
metric_lines = ["| Varian | MAE (µg/m³) | RMSE (µg/m³) |", "| --- | ---: | ---: |"]
metric_lines.extend(
    f"| {key} | {row['mae']:.2f} | {row['rmse']:.2f} |" for key, row in demo_metrics.iterrows()
)
run_label = "SMOKE — cek eksekusi 30 ronde" if SMOKE else "Eksperimen penuh"
demo_summary = f"""# Pavana ML — hasil eksperimen

**Status:** {run_label}; riset retrospektif, belum operasional.

## Pertanyaan dan data
Apakah cuaca, deteksi hotspot, dan indikator upwind membantu memperkirakan PM2.5 CAMS t+6 jam
di satu sel grid sekitar Sungai Raya, Kubu Raya?
Snapshot: {data_audit["aligned"]["rows"]:,} jam dari {data_audit["aligned"]["start_utc"]}
sampai {data_audit["aligned"]["end_utc"]} (UTC). Target CAMS adalah keluaran model;
cuaca adalah reanalisis ERA5; hotspot adalah deteksi satelit, bukan kebakaran terkonfirmasi.
Observasi BMKG hanya konteks dengan arsip pendek; tidak digunakan sebagai target training.

## Metode
A = persistence; B = rata-rata bulan/jam dari train; C = LightGBM riwayat CAMS + cuaca/waktu;
D = C + deteksi/FRP; E = D + upwind. Split kronologis train 2024, validation 2025, test 2026,
dengan jeda enam jam. LightGBM {selected} dipilih hanya dari validation MAE.
Semua varian dinilai pada {len(demo_predictions):,} jam test yang sama. Nilai hilang tetap NaN.

## Hasil
{chr(10).join(metric_lines)}

{takeaway}

## Batas kesimpulan
Hasil belum membuktikan akurasi PM2.5 lapangan, asal asap, atau skill prakiraan live.
Reanalisis ERA5 dan snapshot CAMS tidak memiliki bukti ketersediaan saat prakiraan diterbitkan;
baseline CAMS issue-time tidak dapat diuji. Tidak ada klaim kausal atau interval ketidakpastian.
Arsip observasi stasiun dan prakiraan historis issue-time belum tersedia dalam eksperimen ini.

## Bukti yang disertakan
`report.json`, `metrics.csv`, `test_predictions.parquet`, model C/D/E beserta metadata,
`cloud_run.json`, dan folder `presentation/` berisi grafik serta tabel.
Kode dan data masing-masing dipin di `cloud_run.json`; sumber dataset: https://huggingface.co/datasets/myudak/pavana.
"""
(presentation_dir / "ringkasan_riset.md").write_text(demo_summary, encoding="utf-8")
display(Markdown(demo_summary))

## Artifact

Model, prediksi, evaluasi, audit, grafik, kontribusi SHAP, dan dataset Lab ML disimpan dalam satu
ZIP. Dataset web hanya berisi kasus historis yang telah dihitung; tidak digunakan untuk serving
prakiraan live.

In [ ]:
import importlib.metadata
import platform
import shutil
from datetime import UTC, datetime


def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


cloud_run = {
    "recorded_at_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "notebook": "00_cloud_setup.ipynb",
    "code": {"ref": CODE_REF, "revision": CODE_REVISION, "resolved_from_head": RESOLVE_CODE_HEAD},
    "data": {
        "requested_revision": stored["requested_revision"],
        "resolved_revision": stored["resolved_revision"],
        "manifest_sha256": stored["manifest_sha256"],
    },
    "run": {
        "version": report["version"],
        "directory": str(run["output_dir"]),
        "max_rounds": MAX_ROUNDS,
        "smoke": SMOKE,
    },
    "runtime": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
    },
    "packages": {
        name: package_version(name)
        for name in (
            "pavana-ml",
            "pandas",
            "numpy",
            "pyarrow",
            "lightgbm",
            "scikit-learn",
            "huggingface-hub",
            "shap",
            "matplotlib",
        )
    },
    "status": report["status"],
}
cloud_path = WORK_ROOT / "cloud_run.json"
cloud_run["presentation"] = {
    "target_kind": "model",
    "selected_variant": selected,
    "first_week_target_start_utc": week["target_valid_at"].min().isoformat(),
    "first_week_target_end_utc": week["target_valid_at"].max().isoformat(),
    "selection": "first 168 consecutive holdout target hours; never chosen by target/error",
    "metrics_recomputed_from_predictions": True,
    "shap_method": "tree_path_dependent",
}
cloud_path.write_text(json.dumps(cloud_run, indent=2) + "\n", encoding="utf-8")
shutil.copy2(cloud_path, run["output_dir"] / "cloud_run.json")
(run["output_dir"] / "data_audit.json").write_text(
    json.dumps(data_audit, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
archive = shutil.make_archive(
    str(WORK_ROOT / f"pavana-{report['version']}"), "zip", run["output_dir"]
)


print()
print("cloud_run.json:", cloud_path)
print("artifact zip  :", archive)


import zipfile

required = {
    "presentation/shap_importance.png",
    "presentation/shap_beeswarm.png",
    "presentation/shap_waterfall.png",
    "research.json",
    "cloud_run.json",
    "data_audit.json",
    "report.json",
    "metrics.csv",
    "test_predictions.parquet",
    "presentation/ringkasan_riset.md",
    "presentation/holdout_errors.png",
    "presentation/first_holdout_week.png",
    "presentation/holdout_metrics.csv",
    "presentation/first_holdout_week.csv",
    *(f"{key}/{name}" for key in "CDE" for name in ("model.txt", "metadata.json")),
}
with zipfile.ZipFile(archive) as zipped:
    if not required.issubset(zipped.namelist()) or zipped.testzip() is not None:
        raise ValueError("ZIP belum lengkap atau ada file rusak; jangan tutup sesi.")
from IPython.display import FileLink

# Kaggle exposes relative files through its output viewer; the Output panel is a fallback.
download_path = Path(archive)
if download_path.is_relative_to(Path.cwd()):
    download_path = download_path.relative_to(Path.cwd())
display(FileLink(str(download_path), result_html_prefix="Unduh ZIP hasil lengkap: "))
print("ZIP diperiksa: grafik, ringkasan, metrik, prediksi, model, dan catatan run lengkap.")

## Interpretasi dan batas penerapan

Hasil perbandingan mencakup baseline yang menang maupun model yang kalah. Kontribusi SHAP
menjelaskan perhitungan model relatif terhadap nilai dasarnya; kontribusi bukan bukti penyebab
polusi. Riwayat CAMS dan fitur cuaca yang berkorelasi dapat berbagi atribusi.

Reanalisis dan medan CAMS tidak membuktikan ketersediaan input saat prakiraan diterbitkan.
Validasi terhadap observasi stasiun, generalisasi wilayah, dan evaluasi interval ketidakpastian
belum dilakukan. Status artifact tetap riset retrospektif dan belum operasional.

### Sumber

- [Snapshot Pavana](https://huggingface.co/datasets/myudak/pavana)
- [CAMS global](https://ads.atmosphere.copernicus.eu/datasets/cams-global-atmospheric-composition-forecasts)
- [SHAP TreeExplainer](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html)